# Azure ML Spark HDFS submission test

This notebook verifies that the local `azure-ai-ml` source serializes an HDFS input as `Hdfs` and that the Azure ML service accepts the Spark job submission. The placeholder HDFS endpoint is intentionally not expected to be reachable, so runtime failure does not mean submission validation failed.

Run this notebook from the repository root on a branch containing the HDFS SDK changes.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
sdk_source = repo_root / "sdk" / "ml" / "azure-ai-ml"
if not sdk_source.exists():
    raise RuntimeError("Start Jupyter from the azure-sdk-for-python repository root.")

sys.path.insert(0, str(sdk_source))

import azure.ai.ml

print("Using azure-ai-ml from:", azure.ai.ml.__file__)

## Configuration

The notebook uses Azure ML serverless Spark resources, so no attached compute or registered environment is required. The HDFS URI can remain a placeholder when testing API acceptance only.

In [ ]:
SUBSCRIPTION_ID = "<subscription-id>"
RESOURCE_GROUP = "<resource-group>"
WORKSPACE_NAME = "<workspace-name>"
SPARK_INSTANCE_TYPE = "Standard_E8S_V3"
SPARK_RUNTIME_VERSION = "3.4.0"

HDFS_URI = "hdfs://namenode.example:8020/path/to/data"
CANCEL_AFTER_ACCEPTANCE = True

required_values = {
    "SUBSCRIPTION_ID": SUBSCRIPTION_ID,
    "RESOURCE_GROUP": RESOURCE_GROUP,
    "WORKSPACE_NAME": WORKSPACE_NAME,
}
missing = [name for name, value in required_values.items() if value.startswith("<")]
if missing:
    raise ValueError(f"Replace the placeholder values for: {', '.join(missing)}")

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)

workspace = ml_client.workspaces.get(WORKSPACE_NAME)
print(f"Connected to workspace: {workspace.name}")

print(f"Using serverless Spark: {SPARK_INSTANCE_TYPE}, runtime {SPARK_RUNTIME_VERSION}")

## Create a minimal Spark entry script

In [ ]:
source_dir = repo_root / ".hdfs_submission_test" / "src"
source_dir.mkdir(parents=True, exist_ok=True)

(source_dir / "main.py").write_text(
    '''import argparse
from pyspark.sql import SparkSession

parser = argparse.ArgumentParser()
parser.add_argument("--input-path", required=True)
args = parser.parse_args()

print(f"Attempting to read HDFS input: {args.input_path}")
spark = SparkSession.builder.getOrCreate()
spark.read.text(args.input_path).limit(1).show(truncate=False)
''',
    encoding="utf-8",
)

print("Created:", source_dir / "main.py")

## Construct and inspect the Spark job

The local assertion must show `Hdfs`. If it fails before submission, the checked-out SDK branch does not contain the required serialization mapping.

In [ ]:
from azure.ai.ml import Input, spark
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.constants._common import InputOutputModes
from azure.ai.ml.constants._job.job import JobType
from azure.ai.ml.entities._job._input_output_helpers import to_rest_dataset_literal_inputs

hdfs_mode = getattr(InputOutputModes, "HDFS", "hdfs")
hdfs_input = Input(type=AssetTypes.URI_FOLDER, path=HDFS_URI, mode=hdfs_mode)

job = spark(
    display_name="spark-hdfs-sdk-submission-test",
    experiment_name="spark-hdfs-sdk-submission-test",
    code=str(source_dir),
    entry={"file": "main.py"},
    args="--input-path ${{inputs.hdfs_input}}",
    inputs={"hdfs_input": hdfs_input},
    resources={
        "instance_type": SPARK_INSTANCE_TYPE,
        "runtime_version": SPARK_RUNTIME_VERSION,
    },
    conf={
        "spark.driver.cores": 1,
        "spark.driver.memory": "2g",
        "spark.executor.cores": 1,
        "spark.executor.memory": "2g",
        "spark.executor.instances": 1,
    },
)

try:
    rest_inputs = to_rest_dataset_literal_inputs({"hdfs_input": hdfs_input}, job_type=JobType.SPARK)
except (KeyError, AttributeError) as error:
    raise RuntimeError(
        "The local SDK does not serialize HDFS mode. Check out the HDFS implementation branch first."
    ) from error

serialized_mode = rest_inputs["hdfs_input"].mode
print("Serialized REST input mode:", serialized_mode)
assert str(serialized_mode) == "Hdfs", f"Expected 'Hdfs', received {serialized_mode!r}"


## Submit and confirm API acceptance

Successful creation of a job record confirms that the service accepted the payload. The run may subsequently fail because the placeholder HDFS host cannot be resolved.

In [ ]:
from azure.core.exceptions import HttpResponseError

try:
    created_job = ml_client.jobs.create_or_update(job)
except HttpResponseError as error:
    print("The Azure ML service rejected the submission.")
    print(error)
    raise

print("Job accepted by Azure ML")
print("Name:", created_job.name)
print("Status:", created_job.status)
print("Studio URL:", created_job.studio_url)

In [ ]:
retrieved_job = ml_client.jobs.get(created_job.name)
retrieved_input = retrieved_job.inputs["hdfs_input"]

print("Retrieved job status:", retrieved_job.status)
print("Retrieved input mode:", retrieved_input.mode)
assert str(retrieved_input.mode).lower() == "hdfs"

if CANCEL_AFTER_ACCEPTANCE:
    try:
        ml_client.jobs.begin_cancel(created_job.name).result()
        print("Cancellation requested to avoid unnecessary compute usage.")
    except HttpResponseError as error:
        print("The job could not be cancelled, possibly because it already reached a terminal state:")
        print(error)

## Result interpretation

- **Local serialization fails:** the SDK changes are incomplete or the wrong branch is checked out.
- **Submission reports that `mode` allows only `ReadOnlyMount`, `ReadWriteMount`, `Download`, `Direct`, `EvalMount`, and `EvalDownload`:** the deployed service contract does not support `Hdfs`; a backend/API specification change is required.
- **Submission returns another HTTP validation error:** another configured resource or job field is invalid.
- **Job is created, then fails reading data:** SDK and service submission support are working; a reachable HDFS endpoint and appropriate network/authentication configuration are still required.
- **Job reads data successfully:** full end-to-end HDFS support is working.